# 六模型 RMSE/MAE 对比

该 notebook 对 6 个模型做统一 eval-only 对比，并输出两套数据集上的指标与最佳模型。

In [2]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from ase.io import read
from sklearn.metrics import mean_squared_error, mean_absolute_error

LOREM_ROOT = Path('/data/home/public/qiuqizhi/LOREM')
LOREM_CODE = LOREM_ROOT / 'lorem'
if str(LOREM_CODE) not in sys.path:
    sys.path.insert(0, str(LOREM_CODE))

from calculator import Calculator


In [3]:
MODEL_DIRS = {
    'SOG-mp2': Path('/data/home/public/qiuqizhi/LOREM/my_experiments/cumulene/lorem-sog-cu30-lr-mp2'),
    'SOG-mp2-run2': Path('/data/home/public/qiuqizhi/LOREM/my_experiments/cumulene/lorem-sog-cu30-lr-mp2-run2'),
    'SOG-mp2-run3': Path('/data/home/public/qiuqizhi/LOREM/my_experiments/cumulene/lorem-sog-cu30-lr-mp2-run3'),
    'CU-mp2': Path('/data/home/public/qiuqizhi/LOREM/my_experiments/cumulene/lorem-cu30-lr-mp2'),
    'CU-mp2-run2': Path('/data/home/public/qiuqizhi/LOREM/my_experiments/cumulene/lorem-cu30-lr-mp2-run2'),
    'SOG-mp1-run2': Path('/data/home/public/qiuqizhi/LOREM/my_experiments/cumulene/lorem-sog-cu30-lr-mp1-run2'),
    'SOG-mp1-run3': Path('/data/home/public/qiuqizhi/LOREM/my_experiments/cumulene/lorem-sog-cu30-lr-mp1-run3'),
    'SOG-mp1-run4': Path('/data/home/public/qiuqizhi/LOREM/my_experiments/cumulene/lorem-sog-cu30-lr-mp1-run4'),
    'CU-mp1': Path('/data/home/public/qiuqizhi/LOREM/my_experiments/cumulene/lorem-cu30-lr-mp1'),
    'CU-mp1-run2': Path('/data/home/public/qiuqizhi/LOREM/my_experiments/cumulene/lorem-cu30-lr-mp1-run2'),
    'SOG-mp2-ldependent': Path('/data/home/public/qiuqizhi/LOREM/my_experiments/cumulene/lorem-sog-cu30-lr-mp2-ldependent'),
    'SOG-mp2-ldependent-run2': Path('/data/home/public/qiuqizhi/LOREM/my_experiments/cumulene/lorem-sog-cu30-lr-mp2-ldependent-run2'),
    'SOG-mp2-ldependent-run3': Path('/data/home/public/qiuqizhi/LOREM/my_experiments/cumulene/lorem-sog-cu30-lr-mp2-ldependent-run3'),
    'SOG-mp2-ldependent-run4': Path('/data/home/public/qiuqizhi/LOREM/my_experiments/cumulene/lorem-sog-cu30-lr-mp2-ldependent-run4'),
    'SOG-mp1-ldependent': Path('/data/home/public/qiuqizhi/LOREM/my_experiments/cumulene/lorem-sog-cu30-lr-mp1-ldependent'),
    'SOG-mp1-ldependent-run2': Path('/data/home/public/qiuqizhi/LOREM/my_experiments/cumulene/lorem-sog-cu30-lr-mp1-ldependent-run2'),
    'SOG-mp1-ldependent-run3': Path('/data/home/public/qiuqizhi/LOREM/my_experiments/cumulene/lorem-sog-cu30-lr-mp1-ldependent-run3'),
}

CKPTS = {k: v / 'run/checkpoints/R2_E+F' for k, v in MODEL_DIRS.items()}
TEST_XYZ = LOREM_ROOT / 'datasets' / 'cumulene_test.xyz'
PROFILE_XYZ = LOREM_ROOT / 'datasets' / 'cumulene_profile.xyz'

for name, ckpt in CKPTS.items():
    for p in [
        ckpt / 'model/model.msgpack',
        ckpt / 'model/model.yaml',
        ckpt / 'model/baseline.yaml',
    ]:
        if not p.exists():
            raise FileNotFoundError(f'[{name}] Missing: {p}')

for p in [TEST_XYZ, PROFILE_XYZ]:
    if not p.exists():
        raise FileNotFoundError(f'Missing dataset: {p}')

print('六模型 checkpoint 已就绪:')
for name, ckpt in CKPTS.items():
    print(f'- {name}: {ckpt}')
print('Test dataset   :', TEST_XYZ)
print('Profile dataset:', PROFILE_XYZ)


六模型 checkpoint 已就绪:
- SOG-mp2: /data/home/public/qiuqizhi/LOREM/my_experiments/cumulene/lorem-sog-cu30-lr-mp2/run/checkpoints/R2_E+F
- SOG-mp2-run2: /data/home/public/qiuqizhi/LOREM/my_experiments/cumulene/lorem-sog-cu30-lr-mp2-run2/run/checkpoints/R2_E+F
- SOG-mp2-run3: /data/home/public/qiuqizhi/LOREM/my_experiments/cumulene/lorem-sog-cu30-lr-mp2-run3/run/checkpoints/R2_E+F
- CU-mp2: /data/home/public/qiuqizhi/LOREM/my_experiments/cumulene/lorem-cu30-lr-mp2/run/checkpoints/R2_E+F
- CU-mp2-run2: /data/home/public/qiuqizhi/LOREM/my_experiments/cumulene/lorem-cu30-lr-mp2-run2/run/checkpoints/R2_E+F
- SOG-mp1-run2: /data/home/public/qiuqizhi/LOREM/my_experiments/cumulene/lorem-sog-cu30-lr-mp1-run2/run/checkpoints/R2_E+F
- SOG-mp1-run3: /data/home/public/qiuqizhi/LOREM/my_experiments/cumulene/lorem-sog-cu30-lr-mp1-run3/run/checkpoints/R2_E+F
- SOG-mp1-run4: /data/home/public/qiuqizhi/LOREM/my_experiments/cumulene/lorem-sog-cu30-lr-mp1-run4/run/checkpoints/R2_E+F
- CU-mp1: /data/home/publi

In [4]:
def eval_checkpoint_on_xyz(ckpt_dir: Path, xyz_path: Path, add_offset: bool = True):
    calc = Calculator.from_checkpoint(ckpt_dir, add_offset=add_offset)
    systems = read(xyz_path, index=':')

    e_ref = np.array([s.get_potential_energy() / len(s) for s in systems], dtype=float)
    f_ref = np.concatenate([s.get_forces() for s in systems], axis=0)

    for s in systems:
        s.calc = calc

    e_pred = np.array([s.get_potential_energy() / len(s) for s in systems], dtype=float)
    f_pred = np.concatenate([s.get_forces() for s in systems], axis=0)

    return {
        'energy_rmse_meV_per_atom': float(np.sqrt(mean_squared_error(e_ref, e_pred)) * 1000.0),
        'energy_mae_meV_per_atom': float(mean_absolute_error(e_ref, e_pred) * 1000.0),
        'force_rmse_meV_per_A': float(np.sqrt(mean_squared_error(f_ref, f_pred)) * 1000.0),
        'force_mae_meV_per_A': float(mean_absolute_error(f_ref, f_pred) * 1000.0),
    }


def eval_suite(model_ckpts: dict, xyz_path: Path, add_offset: bool = True):
    rows = []
    for name, ckpt in model_ckpts.items():
        metrics = eval_checkpoint_on_xyz(ckpt, xyz_path, add_offset=add_offset)
        rows.append({'model': name, **metrics})
    return pd.DataFrame(rows)


metrics_cols = [
    'energy_mae_meV_per_atom',
    'energy_rmse_meV_per_atom',
    'force_mae_meV_per_A',
    'force_rmse_meV_per_A',
]

# 在 test 数据集上比较
test_df = eval_suite(CKPTS, TEST_XYZ, add_offset=True)
print('=== model eval on cumulene_test.xyz ===')
display(test_df.sort_values('energy_rmse_meV_per_atom'))

# 在 profile 数据集上比较
profile_df = eval_suite(CKPTS, PROFILE_XYZ, add_offset=True)
print('=== model eval on cumulene_profile.xyz ===')
display(profile_df.sort_values('energy_rmse_meV_per_atom'))

best_test = {m: test_df.loc[test_df[m].idxmin(), ['model', m]].to_dict() for m in metrics_cols}
best_profile = {m: profile_df.loc[profile_df[m].idxmin(), ['model', m]].to_dict() for m in metrics_cols}

print('--- Best on cumulene_test.xyz ---')
for m, v in best_test.items():
    print(f"{m}: {v['model']} ({v[m]:.6f})")

print('--- Best on cumulene_profile.xyz ---')
for m, v in best_profile.items():
    print(f"{m}: {v['model']} ({v[m]:.6f})")


=== model eval on cumulene_test.xyz ===


,model,energy_rmse_meV_per_atom,energy_mae_meV_per_atom,force_rmse_meV_per_A,force_mae_meV_per_A
3,CU-mp2,2.927395,0.556543,44.295037,13.697862
2,SOG-mp2-run3,2.950813,1.031838,47.236225,18.337405
0,SOG-mp2,2.955400,0.935261,47.255144,17.848402
12,SOG-mp2-ldependent-run3,2.984267,1.001043,44.936697,16.472249
11,SOG-mp2-ldependent-run2,3.019130,0.780049,44.782023,18.095223
10,SOG-mp2-ldependent,3.052667,0.708405,46.926211,17.092449
13,SOG-mp2-ldependent-run4,3.061785,1.034125,48.140561,22.306639
1,SOG-mp2-run2,3.101105,0.853059,43.872217,17.540020
14,SOG-mp1-ldependent,3.357204,1.093084,58.795305,27.021008
16,SOG-mp1-ldependent-run3,3.678272,1.337042,68.098696,33.541264


=== model eval on cumulene_profile.xyz ===


,model,energy_rmse_meV_per_atom,energy_mae_meV_per_atom,force_rmse_meV_per_A,force_mae_meV_per_A
10,SOG-mp2-ldependent,0.240248,0.218824,20.552905,8.165022
0,SOG-mp2,0.431762,0.273755,13.786904,6.158384
3,CU-mp2,0.447121,0.178149,38.787421,5.935914
1,SOG-mp2-run2,0.605090,0.505119,23.177547,6.295758
11,SOG-mp2-ldependent-run2,0.719471,0.625139,26.511461,8.703228
16,SOG-mp1-ldependent-run3,0.740844,0.406179,47.559509,13.270274
14,SOG-mp1-ldependent,0.741743,0.660447,30.479319,12.520116
13,SOG-mp2-ldependent-run4,0.913559,0.892361,15.818652,5.784795
6,SOG-mp1-run3,0.950316,0.684117,34.048580,14.679317
12,SOG-mp2-ldependent-run3,1.106805,0.957422,51.638423,8.392448


--- Best on cumulene_test.xyz ---
energy_mae_meV_per_atom: CU-mp2 (0.556543)
energy_rmse_meV_per_atom: CU-mp2 (2.927395)
force_mae_meV_per_A: CU-mp2 (13.697862)
force_rmse_meV_per_A: SOG-mp2-run2 (43.872217)
--- Best on cumulene_profile.xyz ---
energy_mae_meV_per_atom: CU-mp2 (0.178149)
energy_rmse_meV_per_atom: SOG-mp2-ldependent (0.240248)
force_mae_meV_per_A: SOG-mp2-ldependent-run4 (5.784795)
force_rmse_meV_per_A: SOG-mp2 (13.786904)
